# 20b 거리뷰 분할 — 사전학습 DeepLabV3+ 에서 Segment Anything 까지

<a href="https://colab.research.google.com/github/kloud80/urban_datamining_26/blob/main/W06_%EC%9D%B4%EB%AF%B8%EC%A7%80%EB%B6%84%EC%84%9D1-2_%ED%83%90%EC%A7%80%EC%99%80%EB%B6%84%ED%95%A0/notebooks/20b%20%EA%B1%B0%EB%A6%AC%EB%B7%B0%EB%B6%84%ED%95%A0_DeepLab_SAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

`20 Unet` 에서는 항공 영상으로 U-Net 을 **직접 학습**시켰다. 이번에는 학습을 하지 않는다.
도시공학 연구실에서 실제로 쓰던 방식 그대로 — **이미 학습된 분할 모델을 불러와 거리 사진에 씌우고, 픽셀 비율을 도시 지표로 바꾼다.**

| 단계 | 모델 | 하는 일 |
|---|---|---|
| 1 | DeepLabV3+ (Cityscapes 사전학습) | 픽셀마다 19개 클래스 중 하나를 칠한다 |
| 2 | — | 클래스 비율 → 녹시율(GVI)·하늘 개방감·건물 둘러싸임 |
| 3 | Segment Anything (SAM) | 점 하나를 찍으면 그 물체의 경계를 잘라낸다 (이름은 모른다) |
| 4 | DeepLab + SAM | SAM 의 경계에 DeepLab 의 이름을 붙인다 |

### 왜 이 모델인가
한양대 도시공학과 이수기 교수 연구실의 거리 환경 연구들이 Google Street View 사진에 이 모델을 씌워 썼다.

* Ki, Kim & Lee (2021) 『국토계획』 56(4) — 서울 420개 동, 사진 13만 8,930장으로 동별 녹시율을 냈다 (DeepLabV3+ · Cityscapes)
* Hipp, Lee, Ki & Kim (2022) *Journal of Quantitative Criminology* 38(3) — 가로 구간의 거리 요소와 범죄.
  본문: *"we employed the Deeplabv3+ model (82.1% mIoU) … pre-trained with the 'Cityscapes' dataset"*
* Kim, Ki, Osutei, Lee & Hipp (2023) *Journal of Geographical Systems* — 과거 Street View 로 동네 변화 추적.
  배포본 출처로 `github.com/lexfridman/mit-deep-learning` 을 밝혔다.
* Ki & Lee (2021) *Landscape and Urban Planning* 205 — 서울 주민 2,350명의 보행 시간과 녹시율. 녹시율이 기존 녹지 변수보다 보행 시간과 더 밀접했다.

이 노트북은 그 배포본(TensorFlow 공식 model zoo 의 `xception65_cityscapes_trainfine`)을 그대로 쓴다.

> 런타임: CPU 로도 돈다 (한 장에 10~20초). SAM 부분은 GPU 가 있으면 빠르다.

## 1. 준비

In [ ]:
import os, tarfile, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
print('TensorFlow', tf.__version__)

# 한글 폰트 — 그래프 글자가 네모로 깨지지 않게 (Colab 은 나눔고딕을 설치, 윈도는 맑은 고딕)
import platform
import matplotlib.font_manager as fm
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        os.system('apt-get -qq install -y fonts-nanum > /dev/null')
    if os.path.exists(nanum):
        fm.fontManager.addfont(nanum)
        plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

## 2. 사전학습 모델 내려받기

2018년에 구글이 공개한 DeepLabV3+ 체크포인트다. 독일 등 50개 도시의 차량 시점 거리 사진(Cityscapes, 픽셀 단위 정밀 라벨 5,000장)으로 학습됐다.

* `xception65_cityscapes_trainfine` — 논문들이 쓴 것. 정확하지만 무겁다 (약 440MB)
* `mobilenetv2_coco_cityscapes_trainfine` — 가볍고 빠르다 (약 9MB). 사진이 많을 때

In [ ]:
MODEL_NAME = 'xception65_cityscapes_trainfine'
# MODEL_NAME = 'mobilenetv2_coco_cityscapes_trainfine'

URLS = {
    'xception65_cityscapes_trainfine':       'deeplabv3_cityscapes_train_2018_02_06.tar.gz',
    'mobilenetv2_coco_cityscapes_trainfine': 'deeplabv3_mnv2_cityscapes_train_2018_02_05.tar.gz',
}
TARBALL = MODEL_NAME + '.tar.gz'
if not os.path.exists(TARBALL):
    print('내려받는 중... (xception65 는 1~2분)')
    urllib.request.urlretrieve('http://download.tensorflow.org/models/' + URLS[MODEL_NAME], TARBALL)
print('완료', round(os.path.getsize(TARBALL) / 1e6), 'MB')

## 3. 모델 불러오기

이 모델은 TensorFlow 1 시절의 **frozen graph** (가중치가 상수로 굳은 계산 그래프)다.
TF2 에서는 `tf.compat.v1.wrap_function` 으로 감싸면 함수처럼 부를 수 있다.

* 입력 `ImageTensor:0` — `[1, 높이, 너비, 3]` uint8 사진 (긴 변 513px 이하)
* 출력 `SemanticPredictions:0` — `[1, 높이, 너비]` 픽셀마다 0~18 클래스 번호

In [ ]:
with tarfile.open(TARBALL) as tar:
    member = next(m for m in tar.getmembers() if 'frozen_inference_graph' in m.name)
    graph_def = tf.compat.v1.GraphDef.FromString(tar.extractfile(member).read())

_wrapped = tf.compat.v1.wrap_function(lambda: tf.compat.v1.import_graph_def(graph_def, name=''), [])
deeplab = _wrapped.prune('ImageTensor:0', 'SemanticPredictions:0')

INPUT_SIZE = 513

def segment(img):
    """사진 한 장 → (긴 변 513 으로 줄인 사진, 픽셀별 클래스 번호 배열)"""
    img = img.convert('RGB')
    s = INPUT_SIZE / max(img.size)
    small = img.resize((round(img.width * s), round(img.height * s)), Image.LANCZOS)
    seg = deeplab(tf.constant(np.asarray(small)[None]))[0].numpy()
    return small, seg

print('모델 준비 완료')

## 4. 클래스 19개와 색

Cityscapes 의 19개 클래스. 이 목록 **밖의 것은 이 모델이 영원히 못 본다** — 간판·벤치·화단·자전거 거치대는 없다.

In [ ]:
CLASSES = ['road', 'sidewalk', 'building', 'wall', 'fence', 'pole', 'traffic light', 'traffic sign',
           'vegetation', 'terrain', 'sky', 'person', 'rider', 'car', 'truck', 'bus', 'train',
           'motorcycle', 'bicycle']
KOR = ['차도', '보도', '건물', '벽', '울타리', '기둥', '신호등', '표지판', '식생', '지면(풀밭)', '하늘',
       '사람', '탑승자', '승용차', '트럭', '버스', '기차', '오토바이', '자전거']
PALETTE = np.array([
    [128, 64,128],[244, 35,232],[ 70, 70, 70],[102,102,156],[190,153,153],[153,153,153],
    [250,170, 30],[220,220,  0],[107,142, 35],[152,251,152],[ 70,130,180],[220, 20, 60],
    [255,  0,  0],[  0,  0,142],[  0,  0, 70],[  0, 60,100],[  0, 80,100],[  0,  0,230],
    [119, 11, 32]], dtype=np.uint8)

## 5. 거리 사진 한 장

예제는 서울 강남 주거지의 가로수 길이다 (Wikimedia Commons, 퍼블릭 도메인).
**자기 사진을 쓰려면** 왼쪽 파일 탭에 올리고 `IMG_PATH` 만 바꾸면 된다. 사람 눈높이에서 길 방향으로 찍은 사진일수록 연구의 Street View 와 비슷하다.

In [ ]:
IMG_URL = ('https://upload.wikimedia.org/wikipedia/commons/thumb/c/cc/'
           'A_street_in_Gangnam_residential_area%2C_Seoul.jpg/'
           '1280px-A_street_in_Gangnam_residential_area%2C_Seoul.jpg')
IMG_PATH = 'street.jpg'
if not os.path.exists(IMG_PATH):
    req = urllib.request.Request(IMG_URL, headers={'User-Agent': 'urban-datamining-course/1.0'})
    with urllib.request.urlopen(req) as r, open(IMG_PATH, 'wb') as f:
        f.write(r.read())

photo = Image.open(IMG_PATH)
small, seg = segment(photo)
print('입력', photo.size, '→ 모델', small.size, '/ 출력', seg.shape)

## 6. 결과 보기

In [ ]:
from matplotlib.patches import Patch

color = PALETTE[seg]
blend = (0.45 * np.asarray(small) + 0.55 * color).astype(np.uint8)

fig, ax = plt.subplots(1, 3, figsize=(20, 5.5))
for a, im, t in zip(ax, [small, color, blend], ['원본', '분할 결과', '겹쳐 보기']):
    a.imshow(im); a.set_title(t, fontsize=15); a.axis('off')
present = np.unique(seg)
fig.legend(handles=[Patch(color=PALETTE[i] / 255, label=KOR[i]) for i in present],
           loc='lower center', ncol=len(present), fontsize=12, frameon=False)
plt.tight_layout(rect=(0, 0.07, 1, 1)); plt.show()

## 7. 픽셀 비율 → 도시 지표

연구에서 쓰는 지표는 결국 **"사진 속 몇 %가 그 클래스인가"** 다.

* **녹시율 GVI** (Green View Index) = 식생 픽셀 / 전체 픽셀 — 보행자 눈에 보이는 초록의 양
* **하늘 개방감** (Sky View) = 하늘 픽셀 비율 — 높은 건물이 많을수록 낮다
* **건물 둘러싸임** (Enclosure) = 건물 + 벽 비율
* **보도 / 차도** = 보도 비율 / 차도 비율

In [ ]:
share = np.bincount(seg.ravel(), minlength=19) / seg.size * 100

order = np.argsort(share)[::-1]
order = order[share[order] >= 0.1]
plt.figure(figsize=(10, 0.45 * len(order) + 1))
plt.barh([KOR[i] for i in order][::-1], share[order][::-1], color=PALETTE[order][::-1] / 255)
for y, v in enumerate(share[order][::-1]):
    plt.text(v + 0.5, y, f'{v:.1f}%', va='center')
plt.xlabel('사진에서 차지하는 비율 (%)'); plt.title('클래스별 픽셀 비율'); plt.show()

def street_index(seg):
    s = np.bincount(seg.ravel(), minlength=19) / seg.size
    return {
        '녹시율 GVI':   s[CLASSES.index('vegetation')],
        '하늘 개방감':  s[CLASSES.index('sky')],
        '건물 둘러싸임': s[CLASSES.index('building')] + s[CLASSES.index('wall')],
        '보도 비율':    s[CLASSES.index('sidewalk')],
        '차도 비율':    s[CLASSES.index('road')],
    }

for k, v in street_index(seg).items():
    print(f'{k:10s} {v * 100:5.1f} %')

### 연구에서는 한 지점을 여러 장으로 본다
Street View 는 한 지점에서 사방을 볼 수 있다. 연구들은 보통 한 지점의 여러 방향(예: 0°·90°·180°·270°) 사진을 각각 분할해 **평균**을 내고,
그 값을 가로 구간이나 동네 단위로 다시 모아 회귀·기계학습 모델의 **설명변수**로 넣는다.
분할 모델은 답이 아니라 **변수를 만드는 도구**다 — 탐지에서 차량 수가 새 변수였던 것과 같다.

아래 함수는 폴더의 사진을 전부 돌려 표 하나로 만든다.

In [ ]:
import glob
import pandas as pd

def index_table(folder):
    rows = []
    paths = sorted(glob.glob(os.path.join(folder, '*.jp*g')) + glob.glob(os.path.join(folder, '*.png')))
    for p in paths:
        _, s = segment(Image.open(p))
        rows.append({'file': os.path.basename(p), **{k: round(v * 100, 1) for k, v in street_index(s).items()}})
    return pd.DataFrame(rows)

# 예) 사진을 photos/ 폴더에 올린 뒤  index_table('photos')
index_table('.')

## 8. 이 모델의 한계 — 19개 밖은 없다

* **클래스가 고정돼 있다.** 가로수·화단·공원 잔디를 구분하지 못한다(전부 식생/지면). 간판·벤치·노점은 아예 없다.
  → 새 클래스를 원하면 U-Net 처럼 **라벨을 그려서 다시 학습**해야 한다. 20번 노트북의 그 노가다다.
* **학습 데이터가 유럽 도시다.** 서울 골목의 전선, 좁은 이면도로, 상가 간판 앞에서는 틀리기 쉽다.
* **시점이 다르면 틀린다.** 차량 높이에서 찍은 사진으로 배웠다. 항공 사진에 쓰면 엉망이 된다.

그래서 등장한 것이 **"무엇이든 잘라내는"** 모델이다.

## 9. Segment Anything (SAM) — 점 하나로 잘라낸다

Meta 가 2023년 공개. 1,100만 장 사진과 **11억 개 마스크(SA-1B)** 로 학습했다.
클래스 목록이 없다. 대신 **프롬프트**(점·박스)를 주면 그 자리 물체의 경계를 잘라낸다.

> Meta 는 이것을 분할의 **파운데이션 모델**이라 부른다 — 과제마다 다시 학습하지 않고, 지시만 바꿔 어떤 대상에도 쓴다는 뜻이다.
> 단, SAM 은 **경계는 알아도 이름은 모른다.** 잘라낸 것이 나무인지 간판인지는 말해 주지 않는다.

Colab 에는 PyTorch 와 transformers 가 이미 깔려 있다.

In [ ]:
import torch
from transformers import SamModel, SamProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
sam = SamModel.from_pretrained('facebook/sam-vit-base').to(device).eval()
proc = SamProcessor.from_pretrained('facebook/sam-vit-base')
print('SAM 준비 완료 —', device)

### 점을 찍는다
좌표는 `small` 사진 기준 (가로 x, 세로 y) 이다. 기본값은 예제 사진 가운데의 **검은 승용차**와 **왼쪽 가로수 줄기**다.
자기 사진을 쓰면 6번 그림을 보고 좌표를 바꿀 것.

In [ ]:
W, H = small.size
POINTS = {'점 A': (int(W * 0.53), int(H * 0.78)),   # 승용차
          '점 B': (int(W * 0.08), int(H * 0.75))}   # 가로수 줄기

def sam_mask(img, xy):
    inputs = proc(img, input_points=[[list(xy)]], return_tensors='pt').to(device)
    with torch.no_grad():
        out = sam(**inputs, multimask_output=True)
    masks = proc.image_processor.post_process_masks(
        out.pred_masks.cpu(), inputs['original_sizes'].cpu(), inputs['reshaped_input_sizes'].cpu())[0][0]
    best = out.iou_scores[0, 0].argmax().item()     # 후보 3개 중 SAM 스스로 매긴 점수가 가장 높은 것
    return masks[best].numpy(), out.iou_scores[0, 0, best].item()

fig, ax = plt.subplots(1, len(POINTS), figsize=(9 * len(POINTS), 6))
sam_results = {}
for a, (name, xy) in zip(np.atleast_1d(ax), POINTS.items()):
    m, score = sam_mask(small, xy)
    sam_results[name] = m
    a.imshow(small); a.imshow(np.ma.masked_where(~m, m), alpha=0.55, cmap='spring')
    a.scatter(*xy, s=260, c='#00E49B', edgecolors='k', marker='*')
    a.set_title(f'{name} — SAM 점수 {score:.2f}', fontsize=15); a.axis('off')
plt.tight_layout(); plt.show()

## 10. 둘을 합친다 — SAM 의 경계에 DeepLab 의 이름을

SAM 은 **어디까지가 한 물체인지**를 안다. DeepLab 은 **픽셀마다 무엇인지**를 안다.
SAM 마스크 안의 DeepLab 클래스를 세어 다수결로 이름을 붙이면 둘의 장점을 함께 쓸 수 있다.

In [ ]:
for name, m in sam_results.items():
    labels, counts = np.unique(seg[m], return_counts=True)
    top = np.argsort(counts)[::-1][:3]
    desc = ', '.join(f'{KOR[labels[i]]} {counts[i] / m.sum() * 100:.0f}%' for i in top)
    print(f'{name}: 사진의 {m.mean() * 100:.1f}% → {desc}')

## 11. 그다음 — 말로 시킨다 (SAM 2 · SAM 3)

* **SAM 2** (2024.7) — 같은 방식을 **동영상**으로. 한 프레임에 점을 찍으면 끝까지 따라간다. CCTV 에 바로 쓰인다.
* **SAM 3** (2025.11) — 점 대신 **단어**로 시킨다. `"street tree"` 라고 쓰면 사진 속 가로수를 **전부** 찾아 잘라낸다.
  드디어 이름을 알게 됐다. 고정된 19개가 아니라 **아무 명사**나 된다.

SAM 3 가중치는 Hugging Face 에서 **사용 신청 후 승인**을 받아야 받을 수 있다 (`facebook/sam3`).
승인을 받았다면 아래 셀의 주석을 풀어 돌려 볼 것. transformers 버전에 따라 이름이 다를 수 있으니 모델 카드의 예제를 우선한다.

In [ ]:
# from huggingface_hub import login; login()            # 승인받은 계정의 토큰
# from transformers import Sam3Processor, Sam3Model
# p3 = Sam3Processor.from_pretrained('facebook/sam3')
# m3 = Sam3Model.from_pretrained('facebook/sam3').to(device)
# inputs = p3(images=small, text='street tree', return_tensors='pt').to(device)
# with torch.no_grad():
#     out = m3(**inputs)
# res = p3.post_process_instance_segmentation(out, threshold=0.5, mask_threshold=0.5,
#                                             target_sizes=inputs.get('original_sizes').tolist())[0]
# print('찾은 가로수', len(res['masks']), '그루')

## 해 볼 것

1. 자기 동네 길 사진 4장(동서남북)을 올리고 `index_table` 로 녹시율을 비교해 보자. 어느 방향이 가장 초록인가?
2. `MODEL_NAME` 을 mobilenetv2 로 바꾸면 결과가 얼마나 달라지는가? 속도는?
3. SAM 의 점을 간판·현수막에 찍어 보자. DeepLab 은 그것을 무엇이라고 부르는가 — 10번 결과로 확인할 것.
4. 녹시율이 높은 길이 정말 걷기 좋은 길인가? 이 지표가 **놓치는 것**은 무엇인가 (그늘, 계절, 보도 폭…)

### 참고
* Ki, D., Kim, J. H., & Lee, S. (2021). 『국토계획』 56(4), 194–211. doi:10.17208/jkpa.2021.08.56.4.194
* Hipp, J. R., Lee, S., Ki, D., & Kim, J. H. (2022). *Journal of Quantitative Criminology*, 38(3), 537–565. doi:10.1007/s10940-021-09506-9
* Ki, D., & Lee, S. (2021). Analyzing the effects of Green View Index of neighborhood streets on walking time using Google Street View and deep learning. *Landscape and Urban Planning*, 205, 103920.
* Kim, J. H., Ki, D., Osutei, N., Lee, S., & Hipp, J. R. (2023). Beyond visual inspection: capturing neighborhood dynamics with historical Google Street View and deep learning-based semantic segmentation. *Journal of Geographical Systems*.
* Chen, L.-C. et al. (2018). Encoder-Decoder with Atrous Separable Convolution for Semantic Image Segmentation. *ECCV*. (DeepLabV3+)
* Cordts, M. et al. (2016). The Cityscapes Dataset for Semantic Urban Scene Understanding. *CVPR*.
* Kirillov, A. et al. (2023). Segment Anything. *ICCV*.
* 예제 사진: *A street in Gangnam residential area, Seoul* — Wikimedia Commons, Public domain.